In [ ]:
# !pip install tensorflow

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

import matplotlib.pyplot as plt
import numpy as np

from google.colab import files
from PIL import Image, ImageOps

In [ ]:
# Load the MNIST dataset (70,000 grayscale digit images)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values to the range [0, 1]
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape the data to match CNN input: (samples, height, width, channels)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)


In [ ]:
# Improved CNN Model

model = models.Sequential([

    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(10, activation='softmax')
])


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_digit_model.keras",
    monitor='val_accuracy',
    save_best_only=True
)

history = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

print("=" * 40)
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print("=" * 40)

In [ ]:
plt.figure(figsize=(14,5))

# Accuracy Plot
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Plot
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Predictions
y_pred = model.predict(x_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(8,8))
plt.imshow(cm, cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks(np.arange(10))
plt.yticks(np.arange(10))

for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j],
                 ha='center',
                 va='center',
                 color='red',
                 fontsize=8)

plt.colorbar()
plt.show()

In [ ]:
!pip install ipycanvas

import cv2

In [ ]:
# Save the trained model

model.save("handwritten_digit_cnn.keras")

print("✅ Model saved successfully!")

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("handwritten_digit_cnn.keras")

print("✅ Model loaded successfully!")

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import cv2

# ---------------- Canvas ---------------- #

canvas = Canvas(
    width=280,
    height=280,
    sync_image_data=True
)

canvas.fill_style = "black"
canvas.fill_rect(0, 0, 280, 280)

canvas.stroke_style = "white"
canvas.line_width = 18
canvas.line_cap = "round"

drawing = False

def on_mouse_down(x, y):
    global drawing
    drawing = True
    canvas.begin_path()
    canvas.move_to(x, y)

def on_mouse_move(x, y):
    if drawing:
        canvas.line_to(x, y)
        canvas.stroke()

def on_mouse_up(x, y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

# ---------------- Buttons ---------------- #

clear_btn = widgets.Button(
    description="Clear Canvas",
    button_style="danger",
    icon="trash"
)

predict_btn = widgets.Button(
    description="Predict Digit",
    button_style="success",
    icon="check"
)

# ---------------- Clear ---------------- #

def clear_canvas(b):
    canvas.clear()

    canvas.fill_style = "black"
    canvas.fill_rect(0,0,280,280)

    canvas.stroke_style = "white"
    canvas.line_width = 18
    canvas.line_cap = "round"

clear_btn.on_click(clear_canvas)

# ---------------- Prediction ---------------- #

output = widgets.Output()

def predict_digit(b):

    with output:

        output.clear_output()

        img = np.array(canvas.get_image_data())[:, :, :3]

        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        gray = cv2.resize(gray,(28,28))

        gray = gray.astype("float32")/255.0

        plt.figure(figsize=(3,3))
        plt.imshow(gray,cmap="gray")
        plt.title("Processed 28×28 Image")
        plt.axis("off")
        plt.show()

        img_input = gray.reshape(1,28,28,1)

        prediction = loaded_model.predict(
            img_input,
            verbose=0
        )[0]

        predicted_digit = np.argmax(prediction)

        confidence = prediction[predicted_digit]*100

        top3 = prediction.argsort()[-3:][::-1]

        print("="*45)
        print("          PREDICTION RESULT")
        print("="*45)
        print()

        print(f"Predicted Digit : {predicted_digit}")
        print(f"Confidence      : {confidence:.2f}%")

        print("\nTop 3 Predictions")
        print("-"*25)

        for i in top3:
            print(f"{i}  →  {prediction[i]*100:.2f}%")

        print("="*45)

predict_btn.on_click(predict_digit)

# ---------------- Display ---------------- #

print("Draw a digit inside the canvas below.")

display(canvas)
display(widgets.HBox([clear_btn,predict_btn]))
display(output)